# Music Genre Classification System

This notebook implements a deep learning-based system for music genre classification using spectrogram analysis and convolutional neural networks.

## Setup and Dependencies

In [ ]:
# Install required packages
!pip install librosa matplotlib numpy pandas tensorflow sklearn

In [ ]:
import os
import re
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Add, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Configure GPU memory growth to prevent OOM errors
physical_devices = tf.config.list_physical_devices('GPU')
try:
    if physical_devices:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print(f"Found {len(physical_devices)} GPU(s). Memory growth enabled.")
    else:
        print("No GPUs found. Using CPU.")
except Exception as e:
    print(f"Error configuring GPUs: {e}")

## Helper Functions

Implementing utility functions for training visualization and evaluation.

In [ ]:
def plot_history(history, save_path=None):
    """Plot training & validation accuracy and loss values"""
    plt.figure(figsize=(12, 4))
    
    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    
    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Training history plot saved to {save_path}")
    
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, save_path=None, normalized=False):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    if normalized:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = 'Normalized Confusion Matrix'
        save_name = 'normalized_confusion_matrix.png' if save_path is None else save_path
    else:
        title = 'Confusion Matrix'
        save_name = 'confusion_matrix.png' if save_path is None else save_path
    
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title, fontsize=16)
    plt.colorbar()
    
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha='right', fontsize=10)
    plt.yticks(tick_marks, class_names, fontsize=10)
    
    # Add text annotations
    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_cm = format(cm[i, j], '.2f') if normalized else format(cm[i, j], 'd')
            plt.text(j, i, text_cm,
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=9)
    
    plt.tight_layout()
    plt.ylabel('True label', fontsize=12)
    plt.xlabel('Predicted label', fontsize=12)
    
    if save_name:
        plt.savefig(save_name, dpi=300)
        print(f"Confusion matrix saved to {save_name}")
    
    plt.show()

## Audio Processing Functions

Functions for converting audio to spectrograms.

In [ ]:
def create_spectrogram(audio_path, output_dir='spectrograms'):
    """Convert audio file to mel-spectrogram image"""
    os.makedirs(output_dir, exist_ok=True)
    
    filename = os.path.basename(audio_path)
    output_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}.jpg")
    
    # Load audio file
    try:
        y, sr = librosa.load(audio_path)
        
        # Create mel spectrogram
        melspectrogram_array = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        mel = librosa.power_to_db(melspectrogram_array)
        
        # Configure plot
        fig_size = plt.rcParams["figure.figsize"]
        fig_size[0] = float(mel.shape[1]) / float(100)
        fig_size[1] = float(mel.shape[0]) / float(100)
        plt.rcParams["figure.figsize"] = fig_size
        plt.axis('off')
        plt.axes([0., 0., 1., 1.0], frameon=False, xticks=[], yticks=[])
        
        # Plot and save spectrogram
        librosa.display.specshow(mel, cmap='gray_r')
        plt.savefig(output_path, bbox_inches=None, pad_inches=0)
        plt.close()
        
        return output_path
    
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

def slice_spectrogram(spectrogram_path, output_dir='sliced_spectrograms', slice_size=128):
    """Slice spectrogram into square patches"""
    os.makedirs(output_dir, exist_ok=True)
    
    filename = os.path.basename(spectrogram_path)
    base_name = os.path.splitext(filename)[0]
    
    try:
        img = Image.open(spectrogram_path)
        width, height = img.size
        
        # Calculate number of slices
        number_of_samples = int(width / slice_size)
        slices = []
        
        for i in range(number_of_samples):
            start = i * slice_size
            img_temporary = img.crop((start, 0, start + slice_size, slice_size))
            output_path = os.path.join(output_dir, f"{base_name}_slice{i}.jpg")
            img_temporary.save(output_path)
            slices.append(output_path)
        
        return slices
    
    except Exception as e:
        print(f"Error slicing {spectrogram_path}: {e}")
        return []

def process_audio_file(audio_path, create_slices=True):
    """Process audio file: create spectrogram and optionally slice it"""
    print(f"Processing audio file: {audio_path}")
    
    # Create spectrogram
    spectrogram_path = create_spectrogram(audio_path)
    
    if spectrogram_path and create_slices:
        # Slice spectrogram
        slices = slice_spectrogram(spectrogram_path)
        return spectrogram_path, slices
    
    return spectrogram_path, []

## Model Definitions

Model architectures for music genre classification.

In [ ]:
def create_model(input_shape=(128, 128, 1), num_classes=10):
    """Create a ResNet-inspired model for music genre classification"""
    inputs = Input(shape=input_shape)
    
    # Initial convolution block
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # First residual block
    shortcut = x
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(32, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # Second block - increasing filters
    shortcut = Conv2D(64, (1, 1), padding='same')(x)
    shortcut = BatchNormalization()(shortcut)
    
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # Third block
    shortcut = Conv2D(128, (1, 1), padding='same')(x)
    shortcut = BatchNormalization()(shortcut)
    
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # Classification block
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.25)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    # Compile model
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.0001),
        metrics=['accuracy']
    )
    
    return model

def create_improved_model(input_shape=(128, 128, 1), num_classes=10):
    """Create an improved model with squeeze-excitation blocks"""
    inputs = Input(shape=input_shape)
    
    # Initial convolution block
    x = Conv2D(32, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(1e-5))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # First residual block
    shortcut = x
    x = Conv2D(48, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Conv2D(48, (3, 3), padding='same',
               kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # Second block - squeeze and excitation
    shortcut = Conv2D(96, (1, 1), padding='same')(x)
    shortcut = BatchNormalization()(shortcut)
    
    x = Conv2D(96, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Conv2D(96, (3, 3), padding='same',
               kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    
    # Squeeze and Excitation block
    se = tf.keras.layers.GlobalAveragePooling2D()(x)
    se = Dense(96 // 4, activation='relu')(se)
    se = Dense(96, activation='sigmoid')(se)
    se = tf.keras.layers.Reshape((1, 1, 96))(se)
    x = tf.keras.layers.Multiply()([x, se])
    
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # Third block with attention
    shortcut = Conv2D(192, (1, 1), padding='same')(x)
    shortcut = BatchNormalization()(shortcut)
    
    x = Conv2D(192, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Conv2D(192, (3, 3), padding='same',
               kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    
    # Global features
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    
    # Fully connected layers with balanced dropout
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)  # Slightly reduced dropout
    
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    # Compile model with Adam optimizer and slightly lower learning rate
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.00008),
        metrics=['accuracy']
    )
    
    return model

## Dataset Loading and Preprocessing

Load and prepare the GTZAN dataset for music genre classification.

In [ ]:
# Download the GTZAN dataset (if not already downloaded)
!wget -nc http://opihi.cs.uvic.ca/sound/genres.tar.gz
!tar -xzf genres.tar.gz

In [ ]:
def create_dataset(data_path='genres', output_dir='processed_data'):
    """Process audio files to create spectrograms and slices"""
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Dictionary to store data by genre
    genre_data = {}
    
    # Process each genre folder
    for genre in os.listdir(data_path):
        genre_path = os.path.join(data_path, genre)
        if os.path.isdir(genre_path):
            print(f"Processing genre: {genre}")
            genre_data[genre] = []
            
            # Process a subset of files for each genre to keep the notebook manageable
            audio_files = [os.path.join(genre_path, f) for f in os.listdir(genre_path)
                          if f.endswith(".wav") or f.endswith(".au")]
            
            # Process only 10 files per genre to keep the notebook execution quick
            for audio_path in audio_files[:10]:  # Limited to 10 files per genre
                spectrogram_path, slices = process_audio_file(audio_path)
                if slices:
                    genre_data[genre].extend(slices)
    
    return genre_data

def prepare_data(genre_data, test_split=0.2):
    """Prepare data for model training"""
    # Collect all images and labels
    images = []
    labels = []
    label_map = {}
    
    # Assign numeric labels to genres
    for i, genre in enumerate(sorted(genre_data.keys())):
        label_map[genre] = i
    
    # Read and process images
    for genre, slice_paths in genre_data.items():
        label = label_map[genre]
        
        for path in slice_paths:
            try:
                # Load image in grayscale
                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                
                # Resize to ensure consistent dimensions
                img = cv2.resize(img, (128, 128))
                
                # Normalize pixel values
                img = img / 255.0
                
                images.append(img)
                labels.append(label)
            except Exception as e:
                print(f"Error loading {path}: {e}")
    
    # Convert to numpy arrays
    X = np.array(images)
    y = np.array(labels)
    
    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_split, stratify=y, random_state=42)
    
    # Reshape data for CNN (add channel dimension)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)
    
    # Convert labels to categorical
    num_classes = len(label_map)
    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)
    
    return X_train, X_test, y_train, y_test, label_map

In [ ]:
# Create dataset
print("Creating dataset from audio files...")
genre_data = create_dataset()

# Prepare data
print("\nPreparing data for training...")
X_train, X_test, y_train, y_test, label_map = prepare_data(genre_data)

# Print dataset information
print(f"\nTraining data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels shape: {y_test.shape}")
print(f"\nGenre labels: {label_map}")

## Model Training

Train the music genre classification model.

In [ ]:
# Create output directories
output_dir = 'outputs'
model_dir = os.path.join(output_dir, 'models')
plots_dir = os.path.join(output_dir, 'plots')
os.makedirs(model_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

# Create model
num_classes = len(label_map)
model = create_improved_model(input_shape=(128, 128, 1), num_classes=num_classes)

# Print model summary
model.summary()

In [ ]:
# Define callbacks
checkpoint = ModelCheckpoint(
    os.path.join(model_dir, 'best_model.h5'),
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True,
    mode='max'
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=7,
    verbose=1,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=0.000001
)

callbacks = [checkpoint, early_stopping, reduce_lr]

In [ ]:
# Data augmentation
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

# Fit data generator
datagen.fit(X_train)

# Create training generator
train_generator = datagen.flow(
    X_train,
    y_train,
    batch_size=32,
    shuffle=True
)

In [ ]:
# Train model
print("\nStarting training...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // 32,
    epochs=15,  # Reduced for demonstration
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot and save training history
history_path = os.path.join(plots_dir, 'training_history.png')
plot_history(history, save_path=history_path)

# Evaluate model
print("\nEvaluating model on test data:")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f'Test accuracy: {test_acc:.4f}')

# Save final model
model.save(os.path.join(model_dir, 'music_genre_classifier.h5'))
model.save(os.path.join(model_dir, 'music_genre_classifier.keras'))
print(f"Model saved to {model_dir}")

## Model Evaluation

Evaluate the model's performance and visualize results.

In [ ]:
# Generate predictions for confusion matrix
print("Generating predictions for confusion matrix...")
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Get genre names in correct order
genre_names = [genre for genre, idx in sorted([(g, i) for g, i in label_map.items()], key=lambda x: x[1])]

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=genre_names))

# Plot confusion matrices
conf_matrix_path = os.path.join(plots_dir, 'confusion_matrix.png')
norm_conf_matrix_path = os.path.join(plots_dir, 'normalized_confusion_matrix.png')

plot_confusion_matrix(
    y_true_classes, y_pred_classes, 
    genre_names, save_path=conf_matrix_path
)

plot_confusion_matrix(
    y_true_classes, y_pred_classes, 
    genre_names, save_path=norm_conf_matrix_path, 
    normalized=True
)

## Model Prediction for New Audio

Upload and predict the genre of a new audio file.

In [ ]:
from google.colab import files

def predict_genre(model, audio_path, genre_names, top_k=3):
    """Predict genre for a new audio file"""
    print(f"Processing audio file: {os.path.basename(audio_path)}")
    
    # Create spectrogram
    spectrogram_path = create_spectrogram(audio_path, output_dir='temp_spectrograms')
    
    if not spectrogram_path:
        print("Failed to create spectrogram")
        return None
    
    # Slice spectrogram
    slices = slice_spectrogram(spectrogram_path, output_dir='temp_slices')
    
    if not slices:
        print("Failed to create slices")
        return None
    
    # Process all slices
    predictions = []
    for slice_path in slices:
        try:
            # Load image in grayscale
            img = cv2.imread(slice_path, cv2.IMREAD_GRAYSCALE)
            
            # Resize to ensure consistent dimensions
            img = cv2.resize(img, (128, 128))
            
            # Normalize pixel values
            img = img / 255.0
            
            # Add batch and channel dimensions
            img = img.reshape(1, 128, 128, 1)
            
            # Make prediction
            pred = model.predict(img, verbose=0)[0]
            predictions.append(pred)
        except Exception as e:
            print(f"Error processing slice {slice_path}: {e}")
    
    if not predictions:
        print("No valid predictions")
        return None
    
    # Average predictions across all slices
    avg_prediction = np.mean(predictions, axis=0)
    
    # Get top K predictions
    top_indices = np.argsort(avg_prediction)[-top_k:][::-1]
    top_genres = [genre_names[i] for i in top_indices]
    top_probs = [avg_prediction[i] * 100 for i in top_indices]
    
    # Create results
    results = []
    for i in range(top_k):
        results.append((top_genres[i], top_probs[i]))
    
    return results

# Function to upload and predict
def upload_and_predict():
    # Get genre names in correct order
    genre_names = [genre for genre, idx in sorted([(g, i) for g, i in label_map.items()], key=lambda x: x[1])]
    
    print("Please upload an audio file (mp3, wav)")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        file_path = filename
        
        # Predict genre
        results = predict_genre(model, file_path, genre_names)
        
        if results:
            # Display results
            print("\n===== Genre Prediction Results =====")
            print(f"File: {os.path.basename(file_path)}")
            print("\nTop predictions:")
            for i, (genre, prob) in enumerate(results):
                print(f"{i+1}. {genre}: {prob:.2f}%")
                
            # Create a bar chart of predictions
            genres = [r[0] for r in results]
            probs = [r[1] for r in results]
            
            plt.figure(figsize=(10, 5))
            bars = plt.bar(genres, probs, color='skyblue')
            plt.title('Genre Predictions')
            plt.xlabel('Genre')
            plt.ylabel('Confidence (%)')
            plt.xticks(rotation=45)
            
            # Add value labels on top of bars
            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height+1,
                        f'{height:.1f}%', ha='center', va='bottom')
                
            plt.tight_layout()
            plt.show()

In [ ]:
# Upload and predict
upload_and_predict()

## Conclusion

In this notebook, we've implemented a music genre classification system using deep learning and spectrograms. The system converts audio files to visual spectrograms, which are then analyzed by a convolutional neural network to predict the genre.

To improve this system further, you could:
1. Train on a larger dataset
2. Try different model architectures
3. Extract additional audio features
4. Implement ensemble methods
5. Tune hyperparameters